# 08. "최신으로 맞춘다" 가 자료를 나쁘게 만들 때가 있다

> 2026-09-05 · 이동원 · 결론 문서:
> [기능명세 v1.6 버튼 갱신 파이프라인](../../docs/기능명세/version1.6/버튼_갱신_파이프라인.md) ·
> [TIL](../../docs/TIL/이동원/2026-09-05-최신으로-맞추는-것이-자료를-나쁘게-만들-때가-있다.md) ·
> 이슈 [#89 ②](https://github.com/devlee328288/Alpha_Stack/issues/89)

---

## 이 노트북이 답하는 것

손으로 여섯 번 치던 순서를 한 명령(`python -m pipelines.refresh`)으로 묶으면서,
**수정주가 재빌드를 그 안에 넣을 것인가**를 정해야 했습니다.

처음 생각은 당연히 "넣는다" 였습니다. 매일 도는 파이프라인이니 수정주가도 매일 최신으로
맞춰야 할 것 같았습니다. 그런데 그 앞에 물어볼 것이 있었습니다.

> **다시 만들면 자료가 좋아지나?**

재 봤더니 **아니었습니다.** 이 노트북은 그 실측입니다.

| 절 | 무엇을 재나 |
|---|---|
| 1 | FDR 창이 어디에 걸려 있나 |
| 2 | 창이 밀리면 몇 행이 원본에서 근삿값으로 넘어가나 |
| 3 | 새 시세가 반출본을 바꾸나 |
| 4 | 그래서 껐을 때 무엇이 비나 |
| 5 | 파이프라인이 실제로 어떻게 돌았나 |

---

## 0. 준비

DB 는 **읽기 전용**으로 엽니다. 재는 노트북이 쓰기를 열 이유가 없고, 열어 두면 다른
세션의 수집과 잠금을 다툽니다.

In [1]:
import sqlite3
import sys
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

from common.paths import krx_db_path  # noqa: E402

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

conn = sqlite3.connect(f"file:{krx_db_path()}?mode=ro", uri=True)
conn.row_factory = sqlite3.Row
print("DB(읽기전용):", krx_db_path())

DB(읽기전용): C:\Users\kik32\workspace\EST-Camp-AI-Quant\team_project\Alpha_Stack\data\krx_cache.db


---

## 1. FDR 창이 어디에 걸려 있나

FinanceDataReader 는 종목마다 **최근 3,000거래일**만 수정주가를 줍니다. 그 앞은 우리가
앵커에서 이어 붙입니다(`adj_source='chain'`).

`build_adj_prices.py` 가 종목마다 "FDR 이 어디서부터 줬나" 를 `collect_log.cursor` 에
남깁니다. 그 분포를 보면 창이 지금 어디에 걸려 있는지 알 수 있습니다.

In [2]:
창 = pd.read_sql_query(
    """SELECT cursor AS 창시작, COUNT(*) AS 종목수
       FROM collect_log
       WHERE source = 'adj_price' AND cursor IS NOT NULL
       GROUP BY cursor ORDER BY 종목수 DESC LIMIT 6""", conn)
print("FDR 이 준 구간의 시작일 — 상위 6")
display(창)

가장흔한 = 창.iloc[0]["창시작"]
print(f"\n대부분의 종목({창.iloc[0]['종목수']:,}종)에서 창이 {가장흔한} 에 걸려 있습니다.")
print("20100104 로 나오는 종목은 상장 이력이 3,000거래일보다 짧아 전 구간을 FDR 이 준 경우입니다.")

FDR 이 준 구간의 시작일 — 상위 6


,창시작,종목수
0,20140617,1594
1,20100104,520
2,20141224,8
3,20191018,7
4,20141229,7
5,20110517,6



대부분의 종목(1,594종)에서 창이 20140617 에 걸려 있습니다.
20100104 로 나오는 종목은 상장 이력이 3,000거래일보다 짧아 전 구간을 FDR 이 준 경우입니다.


### 반출 구간의 출처 분포

지금 팀에 나가 있는 구간(≤20240830)이 어느 출처로 채워져 있는지 봅니다.

In [3]:
출처 = pd.read_sql_query(
    """SELECT adj_source AS 출처, COUNT(*) AS 행
       FROM daily_price
       WHERE bas_dd <= '20240830' AND adj_close IS NOT NULL
       GROUP BY adj_source ORDER BY 행 DESC""", conn)
출처["비율%"] = (출처["행"] / 출처["행"].sum() * 100).round(2)
display(출처)

,출처,행,비율%
0,fdr,6152027,77.98
1,chain,1692787,21.46
2,fdr+ca_fix,32553,0.41
3,chain+ca_fix,11578,0.15


---

## 2. 🔴 창이 밀리면 몇 행이 원본에서 근삿값으로 넘어가나

**창은 날마다 앞으로 밀립니다.** 오늘 FDR 이 주던 2014년 6월의 행은, 한 달 뒤에는 창 밖으로
밀려나 우리 계산값(`chain`)으로 바뀝니다.

`chain` 은 오차 상한이 **0.39%** 로 실측돼 있습니다
([노트북 04 수정주가를채우다](../02-품질·전처리/04.수정주가를채우다.ipynb)).
즉 **다시 만들 때마다 그만큼이 원본에서 근삿값으로 내려앉습니다.**

거래일 달력으로 창이 20·60·250거래일 밀렸을 때 몇 행이 넘어가는지 셉니다.

In [4]:
달력 = [r[0] for r in conn.execute(
    "SELECT bas_dd FROM trading_calendar WHERE bas_dd >= ? ORDER BY bas_dd", (가장흔한,))]

반출총행 = conn.execute(
    "SELECT COUNT(*) FROM daily_price WHERE bas_dd <= '20240830' AND adj_close IS NOT NULL"
).fetchone()[0]

행들 = []
for n, 설명 in [(20, "약 한 달"), (60, "약 세 달"), (250, "약 한 해")]:
    끝 = 달력[n]
    바뀔행 = conn.execute(
        """SELECT COUNT(*) FROM daily_price
           WHERE adj_source LIKE 'fdr%' AND bas_dd >= ? AND bas_dd < ?""",
        (가장흔한, 끝)).fetchone()[0]
    행들.append({"창이 밀린 거래일": n, "(대략)": 설명, "→ 새 창 시작": 끝,
                "fdr → chain 으로 넘어갈 행": 바뀔행,
                "반출본 대비 %": round(바뀔행 / 반출총행 * 100, 3)})

밀림 = pd.DataFrame(행들)
display(밀림)
print(f"반출본 전체 {반출총행:,}행 기준입니다.")

,창이 밀린 거래일,(대략),→ 새 창 시작,fdr → chain 으로 넘어갈 행,반출본 대비 %
0,20,약 한 달,20140625,11502,0.146
1,60,약 세 달,20140715,38339,0.486
2,250,약 한 해,20141020,158996,2.015


반출본 전체 7,888,945행 기준입니다.


**읽는 법.** 수정주가를 날마다 다시 만들면, 한 달에 11,000행 남짓이 FDR 원본에서 우리
근삿값으로 바뀝니다. 자료가 좋아지는 게 아니라 **조금씩 나빠집니다.**

게다가 값이 바뀌면 `verify_hf_dataset` 이 "재배포 필요" 를 내고, 팀원 넷이 404MB 를 다시
받습니다. 매일 돌리면 매일 그럽니다.

---

## 3. 새 시세가 반출본을 바꾸나 — 안 바꿉니다

반출본은 **개발구간(~20240830)** 만 담습니다. 새로 들어오는 시세는 전부 홀드아웃
구간(20240901~)이라 반출본 밖입니다.

이게 "날마다 수정주가를 다시 만들 이유가 애초에 없다" 의 나머지 절반입니다.

In [5]:
구간 = pd.read_sql_query(
    """SELECT
         CASE WHEN bas_dd <= '20240830' THEN '개발구간 (반출됨)'
              ELSE '홀드아웃 (반출 안 됨)' END AS 구간,
         COUNT(*) AS 행, MIN(bas_dd) AS 시작, MAX(bas_dd) AS 끝
       FROM daily_price GROUP BY 1 ORDER BY 시작""", conn)
display(구간)

,구간,행,시작,끝
0,개발구간 (반출됨),7888945,20100104,20240830
1,홀드아웃 (반출 안 됨),1342993,20240902,20260904


---

## 4. 그래서 껐을 때 무엇이 비나

`--with-adj` 를 끄고 돌리면 그날 받은 시세에 `adj_close` 가 **빈 채로 남습니다.**
어디가 비는지 정확히 봅니다.

In [6]:
최근 = pd.read_sql_query(
    """SELECT bas_dd AS 날짜, COUNT(*) AS 행,
              SUM(CASE WHEN adj_close IS NULL THEN 1 ELSE 0 END) AS 수정주가_결측
       FROM daily_price WHERE bas_dd >= '20260828'
       GROUP BY bas_dd ORDER BY bas_dd""", conn)
display(최근)

반출결측 = conn.execute(
    "SELECT COUNT(*) FROM daily_price WHERE bas_dd <= '20240830' AND adj_close IS NULL"
).fetchone()[0]
print(f"\n반출 구간(≤20240830) 의 수정주가 결측: {반출결측:,}행")

,날짜,행,수정주가_결측
0,20260828,2767,0
1,20260831,2766,0
2,20260901,2765,0
3,20260902,2765,2765
4,20260903,2764,2764
5,20260904,2765,2765



반출 구간(≤20240830) 의 수정주가 결측: 0행


**읽는 법.** 비는 것은 전부 홀드아웃 구간이고, **팀에 나가는 자료에는 영향이 없습니다.**

🔴 다만 이게 "영영 안 켠다" 는 뜻은 아닙니다. 홀드아웃을 열어 최종 평가를 할 때는 그
구간에도 수정주가가 있어야 하므로, 그 직전에 한 번 켭니다.

**날마다가 아니라 필요할 때 켠다** — 이 차이를 문서에 안 적으면 다음 사람이 둘 중 하나로
잘못 읽습니다.

---

## 5. 파이프라인이 실제로 어떻게 돌았나

`ingest_run` · `ingest_run_stage` 에 **시작할 때부터** 남습니다. 두 파이프라인이 같은 표를
나눠 쓰므로 `args` 의 `pipeline` 값으로 가릅니다.

In [7]:
실행 = pd.read_sql_query(
    """SELECT run_id, status AS 상태, started_at AS 시작, finished_at AS 끝
       FROM ingest_run WHERE args LIKE '%"pipeline": "refresh"%'
       ORDER BY started_at DESC LIMIT 4""", conn)
display(실행)

,run_id,상태,시작,끝
0,20260905-164320,ok,2026-09-05T16:43:20+09:00,2026-09-05T16:46:25+09:00
1,20260905-163736,ok,2026-09-05T16:37:36+09:00,2026-09-05T16:42:13+09:00
2,20260905-163648,dry_run,2026-09-05T16:36:49+09:00,2026-09-05T16:36:49+09:00


In [8]:
# 🔴 가장 최근 'ok' 를 그냥 집으면 `--only export` 같은 부분 실행이 잡힌다.
#    보고 싶은 것은 **여섯 단계를 다 거친** 실행이므로 실제로 돈 단계 수로 고른다.
후보 = pd.read_sql_query(
    """SELECT r.run_id,
              SUM(CASE WHEN s.status = 'ok' THEN 1 ELSE 0 END) AS 실제로_돈_단계
       FROM ingest_run r JOIN ingest_run_stage s ON s.run_id = r.run_id
       WHERE r.args LIKE '%"pipeline": "refresh"%' AND r.status = 'ok'
       GROUP BY r.run_id ORDER BY 실제로_돈_단계 DESC, r.run_id DESC""", conn)
display(후보)
열쇠 = 후보.iloc[0]["run_id"]
단계 = pd.read_sql_query(
    """SELECT stage AS 단계, status AS 상태,
              CAST((julianday(finished_at) - julianday(started_at)) * 86400 AS INT) AS 초,
              substr(COALESCE(note, ''), 1, 62) AS 남긴말
       FROM ingest_run_stage WHERE run_id = ? ORDER BY started_at""",
    conn, params=(열쇠,))
print(f"run_id = {열쇠}")
display(단계)
print(f"\n합계 {단계['초'].sum()}초")

,run_id,실제로_돈_단계
0,20260905-163736,3
1,20260905-164320,1


run_id = 20260905-163736


,단계,상태,초,남긴말
0,ingest,ok,19,창 4거래일 · DB 20260901 → 오늘 20260905 · 평일 4일
1,adj,skipped,0,조정 코드가 안 바뀌었다 (--with-adj 로 켠다). 매번 돌리면 FDR 창이...
2,gate,ok,105,리포트: C:\Users\kik32\workspace\EST-Camp-AI-Quan...
3,verify,ok,152,재배포가 필요 없다 — 배포본이 지금 DB·코드와 같다
4,export,skipped,0,판정이 '최신' 이라 반출하지 않는다
5,upload,skipped,0,판정이 '최신' 이라 올리지 않는다



합계 276초


**읽는 법.** `adj` 가 `skipped` 이고 **왜 껐는지가 함께 남아 있습니다.** 이게 요점입니다 —
"안 돌렸다" 와 "돌았는데 실패했다" 는 구별돼야 하고, 까닭이 안 남으면 다음 사람이
"켜는 게 낫겠지" 로 갑니다. 그 반대입니다.

`verify` 가 "재배포가 필요 없다" 로 끝나서 `export` 와 `upload` 가 건너뛰어졌습니다.
**새 시세만으로는 안 올립니다.**

---

## 6. 곁가지 — 진행이 안 보이면 사람은 멈춘 줄 안다

만들고 처음 돌렸을 때 4분 37초 동안 **한 줄도 안 나왔습니다.** 파이썬은 표준출력이
터미널이 아닐 때(파일·파이프로 받을 때) 8KB 씩 모아서 씁니다. 화면은 이 명령을
`subprocess.Popen` 으로 부르므로 정확히 그 경우입니다.

푸는 데 **두 곳**을 건드려야 했습니다. 한 곳만 풀면 나머지가 막습니다.

In [9]:
from pipelines import refresh  # noqa: E402

환경 = refresh._자식_환경()
print("자식에게 주는 환경 —")
for k in ("PYTHONUNBUFFERED", "PYTHONIOENCODING"):
    print(f"   {k} = {환경.get(k)}")

print("\n부모 쪽은 main() 에서 sys.stdout.reconfigure(line_buffering=True) 로 풉니다.")
print("\n단계 순서 —", " → ".join(refresh.STAGES))
print("수집이 수정주가보다 먼저인가:",
      refresh.STAGES.index("ingest") < refresh.STAGES.index("adj"),
      "(새로 받은 시세까지 덮어야 하므로)")

자식에게 주는 환경 —
   PYTHONUNBUFFERED = 1
   PYTHONIOENCODING = utf-8

부모 쪽은 main() 에서 sys.stdout.reconfigure(line_buffering=True) 로 풉니다.

단계 순서 — ingest → adj → gate → verify → export → upload
수집이 수정주가보다 먼저인가: True (새로 받은 시세까지 덮어야 하므로)


---

## 7. 창을 어떻게 세나

고정 `--days 10` 이면 열흘 넘게 안 눌렀을 때 그 앞이 조용히 빕니다. 그리고 **그 구멍은
다음에 눌러도 안 메워집니다** — 창이 언제나 "오늘부터 10일" 이기 때문입니다.

그래서 `daily_price` 의 마지막 날짜에서 거꾸로 잽니다.

In [10]:
마지막 = conn.execute("SELECT MAX(bas_dd) FROM daily_price").fetchone()[0]
print(f"DB 마지막 거래일: {마지막}")

보기 = []
for 안누른날 in (0, 1, 3, 10, 30):
    기준 = date(int(마지막[:4]), int(마지막[4:6]), int(마지막[6:]))
    오늘 = 기준 + timedelta(days=안누른날)
    창수, 설명 = refresh.창을_센다(마지막, 오늘=오늘)
    보기.append({"며칠 안 눌렀나": 안누른날, "그날": f"{오늘:%Y%m%d}",
                "창(거래일)": 창수, "고정 10일이면": 10,
                "구멍 나나": "🔴 난다" if 창수 > 10 else "안 난다"})
display(pd.DataFrame(보기))

DB 마지막 거래일: 20260904


,며칠 안 눌렀나,그날,창(거래일),고정 10일이면,구멍 나나
0,0,20260904,3,10,안 난다
1,1,20260905,3,10,안 난다
2,3,20260907,3,10,안 난다
3,10,20260914,7,10,안 난다
4,30,20261004,21,10,🔴 난다


---

## 8. 정리

| 물음 | 실측한 답 |
|---|---|
| 수정주가를 날마다 다시 만들면 좋아지나 | **아니다.** 한 달에 11,502행(0.146%)이 FDR 원본에서 우리 근삿값으로 내려앉는다 |
| 새 시세가 반출본을 바꾸나 | **안 바꾼다.** 전부 홀드아웃 구간이다 |
| 그럼 영영 안 켜도 되나 | **아니다.** 홀드아웃을 열기 전에 한 번은 켜야 한다 |
| 진행이 왜 안 보였나 | 파이썬이 파이프로 쓸 때 8KB 씩 모은다. 부모·자식 **양쪽**을 풀어야 했다 |
| 고정 창이면 무엇이 문제인가 | 열흘 넘게 안 누르면 구멍이 나고, 그 구멍은 다음에도 안 메워진다 |

> **"최신으로 맞춘다" 는 목적이 아니라 수단입니다.** 무엇을 위해 최신인지 안 물으면,
> 최신으로 맞추는 그 동작이 자료를 갉아먹는 자리가 있습니다.

In [11]:
conn.close()
print("닫았습니다.")

닫았습니다.
